**Motivación del Experimento:** Dado que Isolation Forest obtuvo un rendimiento de detección bajo, el siguiente experimento evalúa **Local Outlier Factor (LOF)**. Debido a su alta complejidad computacional, el algoritmo se aplica bajo restricciones de cómputo.

In [1]:
# Importar las librerías necesarias y funciones de utilidad
import numpy as np
import pandas as pd
from sklearn.neighbors import LocalOutlierFactor

from src.utils import preprocess_for_anomaly_detection

In [2]:
# Cargar los conjuntos de datos
df_train = pd.read_csv('../data/fraudTrain.csv')
df_test = pd.read_csv('../data/fraudTest.csv')

y_train_true = df_train['is_fraud']
X_train_raw = df_train.drop(columns=['is_fraud'])

y_test_true = df_test['is_fraud']
X_test_raw = df_test.drop(columns=['is_fraud'])

print("X_train_raw dimensions:", X_train_raw.shape)
print("X_test_raw dimensions:", X_test_raw.shape)

X_train_raw dimensions: (1296675, 22)
X_test_raw dimensions: (555719, 22)


In [3]:
# Preprocesar train (genera los mapas de frecuencia) y test
X_train, freq_maps = preprocess_for_anomaly_detection(X_train_raw)
X_test, _ = preprocess_for_anomaly_detection(X_test_raw, freq_maps=freq_maps)

print("X_train columns:", X_train.columns.tolist())
print("X_test columns:", X_test.columns.tolist())

X_train columns: ['amt', 'gender', 'city_pop', 'hour_sin', 'hour_cos', 'distance_km', 'age', 'category_freq', 'job_freq', 'state_freq', 'merchant_freq']
X_test columns: ['amt', 'gender', 'city_pop', 'hour_sin', 'hour_cos', 'distance_km', 'age', 'category_freq', 'job_freq', 'state_freq', 'merchant_freq']


### Justificación de Local Outlier Factor (LOF)

Se evaluó LOF como segundo enfoque no supervisado por las siguientes razones:

* **Detección por densidad local:** mientras Isolation Forest aísla anomalías respecto a todo el dataset, LOF compara cada punto contra sus vecinos más cercanos — permite detectar fraudes que no son raros globalmente, pero sí dentro de su contexto inmediato.
* **Complementariedad metodológica:** al usar un principio distinto al de Isolation Forest, permite distinguir si el bajo rendimiento se debe al algoritmo específico o a una limitación general de los enfoques no supervisados en este caso.
* **Costo computacional:** LOF es más costoso que Isolation Forest, por lo que se usó `n_neighbors=20` como balance entre precisión y tiempo de ejecución.

In [4]:
# Tasa de contaminación calculada sobre TRAIN (no combinado con test)
contamination_rate = (y_train_true == 1).sum() / len(y_train_true)
print(f"Exact contamination rate (train): {contamination_rate:.6f}")

# novelty=True permite entrenar con un set y predecir sobre otro nunca visto
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=contamination_rate,
    novelty=True,
    n_jobs=-1
)

# Entrenar SOLO con train
lof.fit(X_train)

# Predecir SOLO sobre test (nunca visto)
preds_lof_test = lof.predict(X_test)
preds_lof_test_binary = [1 if p == -1 else 0 for p in preds_lof_test]

# Evaluar el rendimiento sobre test
total_frauds_test = (y_test_true == 1).sum()
captured_frauds = sum(
    1 for p, r in zip(preds_lof_test_binary, y_test_true) if p == 1 and r == 1
)

print(f"Total fraud cases (test):     {total_frauds_test:,}")
print(f"Frauds detected by LOF:       {captured_frauds:,}")
print(f"Fraud detection rate (Recall): {(captured_frauds / total_frauds_test) * 100:.2f}%")

Exact contamination rate (train): 0.005789


/home/gabriel/Documentos/TecLab/Práctica Profesional/venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


Total fraud cases (test):     2,145
Frauds detected by LOF:       418
Fraud detection rate (Recall): 19.49%


In [5]:
print(type(X_train), type(X_test))
print(X_train.columns.tolist() == X_test.columns.tolist())

<class 'pandas.DataFrame'> <class 'pandas.DataFrame'>
True


**Conclusión Final:**

LOF obtuvo un recall del 19.49% sobre el conjunto de test, superando ampliamente a Isolation Forest (1.96%–2.10%). Esto confirma que el enfoque de densidad local logra capturar mejor ciertos patrones de fraude que el aislamiento geométrico global — sin embargo, el rendimiento sigue siendo muy inferior al del modelo supervisado (XGBoost), que alcanzó una precisión y recall mucho más altos con el umbral optimizado.

Esto refuerza la conclusión central: en la detección de fraudes financieros con etiquetas históricas disponibles, los métodos no supervisados —incluso el mejor de los dos evaluados— muestran un rendimiento limitado, porque identifican anomalías estadísticas en lugar de comportamientos engañosos que simulan intencionalmente transacciones legítimas. Por eso, los modelos supervisados siguen siendo el enfoque preferido para sistemas de detección de fraude en entornos de producción, reservando el enfoque no supervisado como complemento para detectar posibles casos de fraude "zero-day" (con nuevos patrones) no capturados por el modelo principal (XGBoost).